# 02 · Extracción de características

Objetivo: construir la matriz de características que alimentará los modelos.

1. Limpieza y normalización
2. Codificación de variables
3. Selección/ingeniería de características
4. Guardado de la matriz lista para modelado

In [4]:
import pandas as pd

df = pd.read_csv("../data/etiquetas_revision.csv")

print("Dimensiones:", df.shape)
print("Columnas:", df.columns.tolist())
display(df.head())

Dimensiones: (5013, 10)
Columnas: ['imagen', 'macarons', 'dulce', 'salado', 'amargo', 'acido', 'umami', 'picante', 'revisado_por', 'estado_revision']


,imagen,macarons,dulce,salado,amargo,acido,umami,picante,revisado_por,estado_revision
0,huevos_rancheros_001.jpg,huevos_rancheros,0.0,1.0,0.0,0.0,0.0,1.0,fragoso,listo
1,pizza_001.jpg,pizza,0.0,1.0,0.0,0.0,1.0,0.0,fragoso,listo
2,chocolate_cake_001.jpg,chocolate_cake,1.0,0.0,1.0,0.0,0.0,0.0,landan,listo
3,miso_soup_001.jpg,miso_soup,0.0,1.0,0.0,0.0,1.0,0.0,cami,listo
4,bruschetta_001.jpg,bruschetta,0.0,1.0,0.0,1.0,1.0,0.0,landan,listo


In [8]:

from pathlib import Path
import pandas as pd
import numpy as np

# Cargar las etiquetas
df = pd.read_csv("../data/etiquetas_revision.csv")

# Definir las cinco etiquetas de sabores
columnas_sabores = [
    "dulce",
    "salado",
    "amargo",
    "acido",
    "umami"
]

# Eliminar la columna picante si existe
if "picante" in df.columns:
    df = df.drop(columns=["picante"])

# Revisar nombres de imagen vacíos antes de limpiar
print("Nombres de imagen vacíos:", df["imagen"].isna().sum())

# Normalizar los estados:
# quitar espacios y convertir todo a minúsculas
df["estado_revision"] = (
    df["estado_revision"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Incluir todos los registros listos,
# independientemente de mayúsculas o espacios
estados_validos = ["listo", "Listos","pendiente","Listo"]

df = df[
    df["estado_revision"].isin(estados_validos)
].copy()

# Eliminar registros sin nombre de imagen
df = df.dropna(subset=["imagen"]).copy()

# Limpiar nombres de imagen
df["imagen"] = df["imagen"].astype(str).str.strip()

# Eliminar nombres vacíos
df = df[df["imagen"] != ""].copy()

# Convertir las etiquetas de sabores a valores numéricos
for columna in columnas_sabores:
    df[columna] = pd.to_numeric(
        df[columna],
        errors="coerce"
    ).fillna(0).astype(int)

# Construir las rutas de las imágenes
carpeta_imagenes = Path("../data/imagenes")

df["ruta"] = df["imagen"].apply(
    lambda nombre: str(carpeta_imagenes / nombre)
)

# Verificar qué imágenes existen
df["existe"] = df["ruta"].apply(
    lambda ruta: Path(ruta).is_file()
)

# Mostrar resultados
print("\nRegistros listos:", len(df))
print("Imágenes encontradas:", df["existe"].sum())
print("Imágenes faltantes:", (~df["existe"]).sum())

# Mostrar distribución de sabores
print("\nDistribución de sabores:")
display(
    df[columnas_sabores]
    .sum()
    .sort_values(ascending=False)
)

# Mostrar ejemplos de los datos
print("\nEjemplos:")
display(
    df[["imagen", "estado_revision", "ruta", "existe"]
       + columnas_sabores].head()
)

# Revisar si hay filas sin ningún sabor asignado
sin_sabores = df[columnas_sabores].sum(axis=1) == 0

print("\nRegistros sin ningún sabor:", sin_sabores.sum())

# Revisar si hay valores distintos de 0 y 1
for columna in columnas_sabores:
    valores_invalidos = ~df[columna].isin([0, 1])
    if valores_invalidos.any():
        print(
            f"Valores diferentes de 0 y 1 en {columna}:",
            df.loc[valores_invalidos, columna].unique()
        )


Nombres de imagen vacíos: 13

Registros listos: 4951
Imágenes encontradas: 4951
Imágenes faltantes: 0

Distribución de sabores:


salado    3858
umami     3223
dulce     1781
acido     1737
amargo     198
dtype: int64


Ejemplos:


,imagen,estado_revision,ruta,existe,dulce,salado,amargo,acido,umami
0,huevos_rancheros_001.jpg,listo,..\data\imagenes\huevos_rancheros_001.jpg,True,0,1,0,0,0
1,pizza_001.jpg,listo,..\data\imagenes\pizza_001.jpg,True,0,1,0,0,1
2,chocolate_cake_001.jpg,listo,..\data\imagenes\chocolate_cake_001.jpg,True,1,0,1,0,0
3,miso_soup_001.jpg,listo,..\data\imagenes\miso_soup_001.jpg,True,0,1,0,0,1
4,bruschetta_001.jpg,listo,..\data\imagenes\bruschetta_001.jpg,True,0,1,0,1,1



Registros sin ningún sabor: 4


In [9]:

print("Distribución de sabores:")
display(
    df[columnas_sabores]
    .sum()
    .sort_values(ascending=False)
    .rename("Cantidad de imágenes con sabor")
    .to_frame()
)

print("\nCantidad de sabores por imagen:")
display(
    df[columnas_sabores]
    .sum(axis=1)
    .value_counts()
    .sort_index()
    .rename_axis("Número de sabores")
    .to_frame("Cantidad de imágenes")
)


Distribución de sabores:


,Cantidad de imágenes con sabor
salado,3858
umami,3223
dulce,1781
acido,1737
amargo,198



Cantidad de sabores por imagen:


,Cantidad de imágenes
Número de sabores,
0,4
1,1032
2,2377
3,1141
4,397


In [10]:

import sys
from pathlib import Path

# Permitir importar funciones desde la carpeta src
sys.path.insert(0, "../src")

from features import extract_features

# Probar con una sola imagen
ruta_prueba = df.iloc[0]["ruta"]
caracteristicas = extract_features(ruta_prueba)

print("Imagen de prueba:", df.iloc[0]["imagen"])
print("Forma del vector:", caracteristicas.shape)
print("¿Todos los valores son finitos?:", np.isfinite(caracteristicas).all())


Imagen de prueba: huevos_rancheros_001.jpg
Forma del vector: (1814,)
¿Todos los valores son finitos?: True


In [11]:

import numpy as np
import pandas as pd
from pathlib import Path
import sys

# Importar las funciones de extracción
sys.path.insert(0, "../src")
from features import extract_dataset

# Obtener las rutas de las imágenes
rutas = df["ruta"].tolist()

# Extraer las características
X, rutas_validas, errores = extract_dataset(rutas)

# Alinear las etiquetas con las imágenes que sí se procesaron
rutas_validas = [str(Path(ruta)) for ruta in rutas_validas]

df["ruta_normalizada"] = df["ruta"].apply(
    lambda ruta: str(Path(ruta))
)

df_validas = (
    df.set_index("ruta_normalizada")
      .loc[rutas_validas]
      .reset_index()
)

y = df_validas[columnas_sabores].to_numpy(dtype=np.uint8)

# Crear la carpeta de salida
carpeta_salida = Path("../data/procesado")
carpeta_salida.mkdir(parents=True, exist_ok=True)

# Guardar las matrices y las rutas
np.save(carpeta_salida / "X.npy", X)
np.save(carpeta_salida / "y.npy", y)

pd.DataFrame({
    "ruta": rutas_validas
}).to_csv(
    carpeta_salida / "rutas_imagenes.csv",
    index=False
)

# Guardar errores, si los hubo
if errores:
    pd.DataFrame(errores).to_csv(
        carpeta_salida / "errores_extraccion.csv",
        index=False
    )

# Mostrar resultados
print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
print("Imágenes procesadas:", len(rutas_validas))
print("Errores de extracción:", len(errores))
print("¿X contiene valores finitos?:", np.isfinite(X).all())
print("Archivos guardados en:", carpeta_salida.resolve())


Procesadas 100 de 4951 imágenes
Procesadas 200 de 4951 imágenes
Procesadas 300 de 4951 imágenes
Procesadas 400 de 4951 imágenes
Procesadas 500 de 4951 imágenes
Procesadas 600 de 4951 imágenes
Procesadas 700 de 4951 imágenes
Procesadas 800 de 4951 imágenes
Procesadas 900 de 4951 imágenes
Procesadas 1000 de 4951 imágenes
Procesadas 1100 de 4951 imágenes
Procesadas 1200 de 4951 imágenes
Procesadas 1300 de 4951 imágenes
Procesadas 1400 de 4951 imágenes
Procesadas 1500 de 4951 imágenes
Procesadas 1600 de 4951 imágenes
Procesadas 1700 de 4951 imágenes
Procesadas 1800 de 4951 imágenes
Procesadas 1900 de 4951 imágenes
Procesadas 2000 de 4951 imágenes
Procesadas 2100 de 4951 imágenes
Procesadas 2200 de 4951 imágenes
Procesadas 2300 de 4951 imágenes
Procesadas 2400 de 4951 imágenes
Procesadas 2500 de 4951 imágenes
Procesadas 2600 de 4951 imágenes
Procesadas 2700 de 4951 imágenes
Procesadas 2800 de 4951 imágenes
Procesadas 2900 de 4951 imágenes
Procesadas 3000 de 4951 imágenes
Procesadas 3100 de 

In [13]:

import numpy as np
import pandas as pd

# Revisar los valores posibles de cada etiqueta
print("Valores únicos por sabor:")
for i, sabor in enumerate(columnas_sabores):
    print(f"{sabor}: {np.unique(y[:, i])}")

# Contar cuántas imágenes tienen cada sabor
print("\nCantidad de imágenes por sabor:")
display(
    pd.DataFrame({
        "sabor": columnas_sabores,
        "cantidad": y.sum(axis=0).astype(int)
    })
)

# Revisar imágenes sin etiquetas
sin_etiquetas = (y.sum(axis=1) == 0).sum()
print("\nImágenes sin ningún sabor:", sin_etiquetas)

# Revisar cuántos sabores tiene cada imagen
print("\nNúmero de sabores por imagen:")
print(pd.Series(y.sum(axis=1)).value_counts().sort_index())


Valores únicos por sabor:
dulce: [0 1]
salado: [0 1]
amargo: [0 1]
acido: [0 1]
umami: [0 1]

Cantidad de imágenes por sabor:


,sabor,cantidad
0,dulce,1781
1,salado,3858
2,amargo,198
3,acido,1737
4,umami,3223



Imágenes sin ningún sabor: 4

Número de sabores por imagen:
0       4
1    1032
2    2377
3    1141
4     397
Name: count, dtype: int64


In [14]:

import numpy as np
from sklearn.model_selection import train_test_split

# Cargar las matrices guardadas
X = np.load("../data/procesado/X.npy")
y = np.load("../data/procesado/y.npy")

# Dividir los datos: 70 % entrenamiento, 30 % prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

print("Entrenamiento:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nPrueba:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nCantidad de positivos por sabor:")
for i, sabor in enumerate(columnas_sabores):
    print(
        f"{sabor}: "
        f"entrenamiento={int(y_train[:, i].sum())}, "
        f"prueba={int(y_test[:, i].sum())}"
    )


Entrenamiento:
X_train: (3465, 1814)
y_train: (3465, 5)

Prueba:
X_test: (1486, 1814)
y_test: (1486, 5)

Cantidad de positivos por sabor:
dulce: entrenamiento=1210, prueba=571
salado: entrenamiento=2731, prueba=1127
amargo: entrenamiento=131, prueba=67
acido: entrenamiento=1228, prueba=509
umami: entrenamiento=2296, prueba=927


In [15]:

from pathlib import Path
import numpy as np
import pandas as pd

# Crear carpeta para los conjuntos de entrenamiento y prueba
carpeta_salida = Path("../data/procesado")
carpeta_salida.mkdir(parents=True, exist_ok=True)

# Guardar características y etiquetas
np.save(carpeta_salida / "X_train.npy", X_train)
np.save(carpeta_salida / "X_test.npy", X_test)
np.save(carpeta_salida / "y_train.npy", y_train)
np.save(carpeta_salida / "y_test.npy", y_test)

# Guardar los nombres y el orden de las etiquetas
pd.Series(columnas_sabores, name="sabor").to_csv(
    carpeta_salida / "columnas_sabores.csv",
    index=False
)

print("Conjuntos guardados correctamente.")
print("Archivos:", [p.name for p in carpeta_salida.glob("*.npy")])


Conjuntos guardados correctamente.
Archivos: ['X.npy', 'X_test.npy', 'X_train.npy', 'y.npy', 'y_pred_lr.npy', 'y_pred_rf.npy', 'y_pred_svm.npy', 'y_test.npy', 'y_train.npy']
